# ATLAS on Qwen2.5-7B — 18-way, host `train.csv` only

Training kernel (placeholder `submission.parquet` only). Same **ATLAS** recipe as [komseok0109/ATLAS](https://github.com/komseok0109/ATLAS): a frozen causal LM, LoRA on the attention projections, one learned **control embedding** per author, and a separate LoRA up-projection **B_k** per author. The loss is the next-token negative log-likelihood of each passage under **its own** author's control code. There is no classification head.

The released `ATLAS-Qwen2.5-7B` checkpoint is a 5-family model (`gpt`, `claude`, `gemini`, `mistral`, `llama`). This notebook does **not** load it. It trains those control codes from scratch on the **18 competition labels**, using only host `train.csv` (2,699 passages).

Five prompt folds plus `final`, two GPUs, same job split as [`Qwen3-8B-base-Lora.ipynb`](../Qwen3-8B/Qwen3-8B-base-Lora.ipynb). Fold seed is **0**, so out-of-fold rows line up with the Qwen3 notebooks. Training seed is **42**, the ATLAS default. Five epochs and learning rate `1e-4` match the released ATLAS run.

**Attach on Kaggle (internet off):**

- competition `name-that-llm`
- Kaggle model **Qwen2.5-7B** base (`qwen-lm/qwen2.5`, the `7b` variation — not Instruct)
- Output of [`ATLAS-18way-setting.ipynb`](ATLAS-18way-setting.ipynb) (Add Input → your setting notebook). That folder has `wheels/` with `peft`, `bitsandbytes`, `accelerate`, `transformers`.

Run the setting notebook first, once, with internet ON and no GPU. Then attach it here. The next cell installs those wheels with `--no-index`.

The frozen base is loaded in 4-bit (NF4). A fp16 Qwen2.5-7B is about 14 GB, which fills a 14.6 GB T4 before the first training step. `MAX_LEN` does not fix that: these passages are at most 381 tokens, so 384 and 512 are the same sequences. LoRA, the control embeddings, and the loss are unchanged.


## 0. Install deps from the setting notebook

Looks under `/kaggle/input` for a `wheels/` folder written by [`ATLAS-18way-setting.ipynb`](ATLAS-18way-setting.ipynb), then installs with `--no-index`. Internet stays off.


In [ ]:
import glob
import os
import subprocess
import sys
from pathlib import Path

PKGS = [
    "transformers==4.51.3",
    "bitsandbytes==0.45.5",
    "accelerate==1.6.0",
    "peft==0.15.2",
]


def find_wheel_dir():
    hits = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if not root.exists():
            continue
        for p in root.rglob("wheels"):
            if p.is_dir() and any(p.glob("*.whl")):
                hits.append(p)
    if not hits:
        raise FileNotFoundError(
            "No wheels/ folder. Run ATLAS-18way-setting.ipynb with internet ON, "
            "then Add Input that notebook output on this train kernel."
        )
    return hits[0]


wheel_dir = find_wheel_dir()
print("wheels", wheel_dir, "n=", len(list(wheel_dir.glob("*.whl"))), flush=True)
cmd = [
    sys.executable, "-m", "pip", "install",
    "--no-index", "--find-links", str(wheel_dir),
    "-q", *PKGS,
]
print(subprocess.check_output(cmd, text=True, stderr=subprocess.STDOUT)[-2000:], flush=True)

import bitsandbytes as bnb
from peft import LoraConfig  # noqa: F401
print("bitsandbytes", getattr(bnb, "__version__", "?"), flush=True)
print("peft ok", flush=True)


## Recipe

| Setting | Value |
| --- | --- |
| Base | Qwen2.5-7B, all 28 layers, loaded from `/kaggle/input` (`local_files_only`) |
| LoRA | r=16, α=32, dropout=0.05 on `q_proj`, `k_proj`, `v_proj`, `o_proj` only |
| Per source | 1 control token + its own LoRA up-projection `B_k` (shared down-projection `A`) |
| Loss | `0.3 *` mean next-token NLL under the true author's control code |
| Optim | AdamW, weight decay 0.01, grad clip 1.0, 5% warmup then cosine |
| Precision | 4-bit NF4 frozen base, fp32 trainable, fp16 autocast, GradScaler 256, grad checkpointing |
| Seq length | 512 tokens (ATLAS `max_len`) |
| Train | 5 epochs, lr 1e-4, batch 1, accum 8, no class weights. Host `train.csv` only. |
| Readout | on each held-out fold: argmax of mean token log-prob, and argmax-prod over T in {1, 2, 4, 8} |

Effective batch is 8, not ATLAS's 128. ATLAS used 4 large GPUs at batch 4. A T4 only fits batch 1 of this 7B model, and accumulation 8 keeps a few hundred optimizer steps on 2,699 rows.

**Saved per job** under `lora_out/job_<id>/`:

| Job | Contents |
| --- | --- |
| `0`–`4` | adapter + `atlas_extra.pt` (control embeddings and every `B_k`), plus `oof_scores.npz` on prompts that job never trained on |
| `final` | inference checkpoint, trained on every host prompt |


## Hyperparameters


In [ ]:
SEED = 42
FOLD_SEED = 0
EPOCHS = 5
LR = 1e-4
BATCH = 1
ACCUM = 8
CLASS_WEIGHT = False
JOBS = {
    0: ["0", "2", "4"],
    1: ["1", "3", "final"],
}

# Shared ATLAS recipe.
MAX_LEN = 512
PREFIX_LEN = 1
LAM_SFT = 0.3
WARMUP = 0.05
WD = 0.01
GRAD_CLIP = 1.0
R = 16
ALPHA = 32
LDROP = 0.05
SCORE_BS = 2
LOG_EVERY = 20
OUT_ROOT = "lora_out"

LABELS = [
    "claude-fable-5",
    "claude-opus-4-6",
    "claude-opus-4-8",
    "claude-opus-5",
    "claude-sonnet-5",
    "gemini-3.1-pro-preview",
    "gemini-3.6-flash",
    "gemini-3.7-flash",
    "gemini-3.8-flash",
    "glm-5.3",
    "glm-5.3-flash",
    "gpt-5.5",
    "gpt-5.6-luna",
    "gpt-5.6-sol",
    "gpt-5.6-terra",
    "gpt-6-astra",
    "kimi-k3",
    "qwen3.8-max",
]
assert len(LABELS) == 18 and len(set(LABELS)) == 18

import glob
import json
import os
import pathlib
import subprocess
import sys
import time

import pandas as pd


## Worker

The notebook is one process. Training runs **two processes**, one per T4, the same way as `Qwen3-8B-base-Lora.ipynb`. The cells below write `worker.py`. Run them in order before the launch cell.

The worker never calls Hugging Face. `AutoModelForCausalLM.from_pretrained` and the tokenizer both use `local_files_only=True`.


### 1. Model: control codes, per-source LoRA B, next-token loss


In [ ]:
%%writefile worker.py
"""ATLAS training worker. One process = one GPU.

Recipe (ko2026atlas): frozen Qwen2.5-7B, LoRA on q/k/v/o, one learned control
embedding per source, and a separate LoRA up-projection B_k per source.
Loss is next-token NLL of each passage under its own source's control code.
"""
import importlib.util as _iu
import json
import math
import os
import random
import sys
import time

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# Kaggle ships torchao 0.10, which peft rejects; hide it before importing peft.
_fs = _iu.find_spec
_iu.find_spec = lambda name, *a, **k: None if name.split(".")[0] == "torchao" else _fs(name, *a, **k)

from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import peft.import_utils as _pi

_pi.is_torchao_available = lambda *a, **k: False

try:
    from peft.tuners.lora.layer import Linear as LoraLinear
except ImportError:
    from peft.tuners.lora import Linear as LoraLinear

try:
    from peft import set_peft_model_state_dict
except ImportError:
    from peft.utils.save_and_load import set_peft_model_state_dict

A2_B_INIT_STD = 1e-4
PROD_TEMPS = (1.0, 2.0, 4.0, 8.0)


class _LoraRoute:
    """Passes each row's source index to the per-source LoRA layers."""

    __slots__ = ("index", "uniform_k")

    def __init__(self):
        self.index = None
        self.uniform_k = None

    def set(self, index, uniform_k):
        self.index, self.uniform_k = index, uniform_k


def _is_peft_lora_linear(mod):
    """4-bit peft wraps q/k/v/o as Linear4bit, not lora.layer.Linear."""
    return (
        hasattr(mod, "lora_A")
        and hasattr(mod, "lora_B")
        and hasattr(mod, "scaling")
        and hasattr(mod, "base_layer")
        and not hasattr(mod, "lora_B_k")
    )


class PerSourceLoraLinear:
    """Mixin: peft LoRA linear whose up-projection is a [K, d_out, r] stack, one slice per source."""

    @classmethod
    def install(cls, layer, num_sources, route, init_std):
        adapters = list(layer.lora_A.keys())
        if len(adapters) != 1:
            raise RuntimeError(f"per-source B expects one adapter, got {adapters}")
        name = adapters[0]
        out_f, r = layer.lora_B[name].weight.shape
        if not isinstance(layer, cls):
            layer.__class__ = type(f"{cls.__name__}_{type(layer).__name__}", (cls, type(layer)), {})
        layer._route = route
        layer._adapter_name = name
        w = torch.randn(
            num_sources, out_f, r,
            dtype=layer.lora_B[name].weight.dtype,
            device=layer.lora_B[name].weight.device,
        ) * init_std
        layer.lora_B_k = nn.Parameter(w)
        layer._freeze_shadow_b()
        return layer

    def _freeze_shadow_b(self):
        with torch.no_grad():
            self.lora_B[self._adapter_name].weight.zero_()
        self.lora_B[self._adapter_name].weight.requires_grad_(False)

    def set_adapter(self, adapter_names, inference_mode=False):
        super().set_adapter(adapter_names, inference_mode=inference_mode)
        self._freeze_shadow_b()

    def forward(self, x, *args, **kwargs):
        if hasattr(self, "_check_forward_args"):
            self._check_forward_args(x, *args, **kwargs)
        kwargs.pop("adapter_names", None)
        if getattr(self, "disable_adapters", False):
            if getattr(self, "merged", False):
                self.unmerge()
            return self.base_layer(x, *args, **kwargs)
        result = self.base_layer(x, *args, **kwargs)
        out_dtype = result.dtype
        name = self._adapter_name
        lora_A = self.lora_A[name]
        dropout = self.lora_dropout[name]
        if hasattr(self, "_cast_input_dtype"):
            x = self._cast_input_dtype(x, lora_A.weight.dtype)
        else:
            x = x.to(lora_A.weight.dtype)
        h = lora_A(dropout(x))
        B = self.lora_B_k
        k = self._route.uniform_k
        if k is not None:
            delta = F.linear(h, B[k])
        else:
            idx = self._route.index
            if idx is None or idx.numel() != h.shape[0]:
                raise RuntimeError("per-source LoRA B: row->source route is missing or the wrong length")
            Bn = B.index_select(0, idx)
            delta = torch.bmm(h, Bn.transpose(1, 2))
        scale = self.scaling[name]
        return (result + delta.to(out_dtype) * scale).to(out_dtype)


def masked_mean(x, mask):
    return (x * mask).sum(dim=-1) / mask.sum(dim=-1).clamp(min=1.0)


def doc_score(theta_lp, mask):
    """Mean token log-prob. theta_lp [B, K, L], mask [B, L] -> [B, K]."""
    return masked_mean(theta_lp, mask.unsqueeze(1))


def prod_readout(theta_lp, mask, temperature):
    """ATLAS argmax-prod at one temperature. [B, K, L] -> [B, K]."""
    m = mask > 0.5
    lp = theta_lp.float()
    gap = lp.unsqueeze(2) - lp.unsqueeze(1)
    v = torch.exp(-(torch.relu(-gap) / float(temperature)).sum(dim=2))
    v = v * m.unsqueeze(1)
    return v.sum(dim=2) / m.sum(dim=1, keepdim=True).clamp(min=1)


class ATLASModel(nn.Module):
    def __init__(self, base_path, source_names, lora_r, lora_alpha, lora_dropout, prefix_len, device):
        super().__init__()
        self.source_names = list(source_names)
        self.num_sources = len(self.source_names)
        self.prefix_len = int(prefix_len)
        self.tail_chunks = 4
        self.device_ = device

        self.tokenizer = AutoTokenizer.from_pretrained(base_path, local_files_only=True)
        self.tokenizer.padding_side = "right"
        self.tokenizer.model_max_length = int(1e9)
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        # fp16 7B weights are ~14 GB and leave a T4 with no room for a training step.
        # The frozen base is NF4; LoRA, control embeddings, and the loss stay fp32/fp16.
        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16,
                )
        load_kw = dict(
            quantization_config=quant,
            device_map={"": 0},
            local_files_only=True,
            low_cpu_mem_usage=True,
            attn_implementation="sdpa",
            torch_dtype=torch.float16,
        )
        base = AutoModelForCausalLM.from_pretrained(base_path, **load_kw)
        base.config.use_cache = False
        base = prepare_model_for_kbit_training(
            base,
            use_gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
        )

        self.model = get_peft_model(
            base,
            LoraConfig(
                r=lora_r,
                lora_alpha=lora_alpha,
                lora_dropout=lora_dropout,
                target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
                bias="none",
                task_type="CAUSAL_LM",
            ),
        )

        self._route = _LoraRoute()
        self._per_source_layers = []
        for name, mod in self.model.named_modules():
            if _is_peft_lora_linear(mod):
                PerSourceLoraLinear.install(mod, self.num_sources, self._route, A2_B_INIT_STD)
                self._per_source_layers.append((name, mod))
        if not self._per_source_layers:
            kinds = sorted({type(m).__name__ for _, m in self.model.named_modules()})
            raise RuntimeError(f"no LoRA layers found to convert to per-source B; modules={kinds[:40]}")
        print(f"per-source B on {len(self._per_source_layers)} LoRA layers ({type(self._per_source_layers[0][1]).__name__})", flush=True)

        for p in self.model.parameters():
            if p.requires_grad:
                p.data = p.data.to(device=device, dtype=torch.float32)

        emb = self.model.get_input_embeddings().weight.detach().float()
        mean = emb.mean(dim=0)
        std = float(emb.std())
        init = mean.view(1, 1, -1).repeat(self.num_sources, self.prefix_len, 1)
        init = init + 0.01 * std * torch.randn(
            self.num_sources, self.prefix_len, emb.shape[1], device=emb.device, dtype=torch.float32,
        )
        self.control_embeddings = nn.Parameter(init.to(device=device, dtype=torch.float32))
        free, total = torch.cuda.mem_get_info(device)
        print(f"loaded 4-bit base, free {free / 1024 ** 3:.2f} / {total / 1024 ** 3:.2f} GiB", flush=True)

    def trainable_parameters(self):
        return [p for p in self.parameters() if p.requires_grad]

    def _decoder(self):
        # PeftModel -> Qwen2ForCausalLM -> Qwen2Model, which accepts inputs_embeds.
        base = self.model.get_base_model()
        return base.model if hasattr(base, "model") else base

    def _lm_head(self, hidden):
        return self.model.get_output_embeddings()(hidden)

    def _theta_pass_single(self, input_ids, attention_mask, labels):
        """Token log p(x_t | x_<t, c_y) under each row's own source. [B, L-1]."""
        bsz, length = input_ids.shape
        pfx = self.prefix_len
        tok_emb = self.model.get_input_embeddings()(input_ids)
        ctrl = self.control_embeddings.index_select(0, labels).to(dtype=tok_emb.dtype)
        inputs_emb = torch.cat([ctrl, tok_emb], dim=1)
        full_attn = torch.cat([attention_mask.new_ones(bsz, pfx), attention_mask], dim=1)
        self._route.set(labels, None)
        hidden = self._decoder()(inputs_embeds=inputs_emb, attention_mask=full_attn).last_hidden_state
        hid = hidden[:, pfx:-1, :]
        targets = input_ids[:, 1:].unsqueeze(-1)
        del hidden
        use_ckpt = self.training and torch.is_grad_enabled()

        def tail(h_c, t_c):
            log_s = F.log_softmax(self._lm_head(h_c).float(), dim=-1)
            return log_s.gather(-1, t_c).squeeze(-1)

        n = max(1, min(self.tail_chunks, hid.size(1)))
        parts = []
        for h_c, t_c in zip(hid.chunk(n, dim=1), targets.chunk(n, dim=1)):
            if use_ckpt:
                parts.append(checkpoint(tail, h_c, t_c, use_reentrant=False))
            else:
                parts.append(tail(h_c, t_c))
        return torch.cat(parts, dim=1)

    def compute_losses(self, input_ids, attention_mask, labels, lam_sft, class_w=None):
        if bool((labels >= self.num_sources).any()):
            raise ValueError("a label is outside the source list")
        loss_mask = attention_mask[:, 1:].float()
        true_lp = self._theta_pass_single(input_ids, attention_mask, labels)
        per = -masked_mean(true_lp, loss_mask)
        if class_w is not None:
            per = per * class_w[labels]
        nll = per.mean()
        return {"loss": lam_sft * nll, "L_sft": nll.detach()}

    def save_weights(self, out_dir):
        os.makedirs(out_dir, exist_ok=True)
        self.model.save_pretrained(out_dir)
        self.tokenizer.save_pretrained(os.path.join(out_dir, "tokenizer"))
        extra = {
            "control_embeddings": self.control_embeddings.detach().float().cpu(),
            "source_names": self.source_names,
            "prefix_len": self.prefix_len,
            "per_source_lora_b": True,
            "lora_B_k": {n: m.lora_B_k.detach().float().cpu() for n, m in self._per_source_layers},
        }
        torch.save(extra, os.path.join(out_dir, "atlas_extra.pt"))

    def load_weights(self, ckpt_dir):
        extra_path = os.path.join(ckpt_dir, "atlas_extra.pt")
        extra = torch.load(extra_path, map_location="cpu", weights_only=False)
        if list(extra["source_names"]) != self.source_names:
            raise ValueError(
                f"checkpoint sources {extra['source_names']} != this run {self.source_names}"
            )
        st_path = os.path.join(ckpt_dir, "adapter_model.safetensors")
        if os.path.exists(st_path):
            from safetensors.torch import load_file
            state = load_file(st_path)
        else:
            state = torch.load(os.path.join(ckpt_dir, "adapter_model.bin"), map_location="cpu", weights_only=False)
        state = {k: v for k, v in state.items() if "lora_B_k" not in k}
        set_peft_model_state_dict(self.model, state)
        saved = extra["control_embeddings"]
        if saved.shape != self.control_embeddings.shape:
            raise ValueError(
                f"control_embeddings {tuple(saved.shape)} != {tuple(self.control_embeddings.shape)}"
            )
        with torch.no_grad():
            self.control_embeddings.copy_(saved.to(self.control_embeddings.dtype))
            saved_b = extra["lora_B_k"]
            for n, m in self._per_source_layers:
                if n not in saved_b:
                    raise KeyError(f"lora_B_k missing layer {n}")
                if tuple(saved_b[n].shape) != tuple(m.lora_B_k.shape):
                    raise ValueError(f"lora_B_k[{n}] {tuple(saved_b[n].shape)} != {tuple(m.lora_B_k.shape)}")
                m.lora_B_k.copy_(saved_b[n].to(m.lora_B_k.dtype))
                m._freeze_shadow_b()
        for p in self.model.parameters():
            if p.requires_grad:
                p.data = p.data.float()
        self.control_embeddings.data = self.control_embeddings.data.float()


### 2. Data, train loop, fold scores, save

Appends the training loop. Each optimizer step is `ACCUM` microbatches of batch size 1. After the last epoch, fold jobs score the held-out prompts under all 18 control codes and write `oof_scores.npz`. `final` trains on every row and does not write a score.


In [ ]:
%%writefile -a worker.py


def bal_acc(y_true, y_pred, n_classes):
    scores = []
    for c in range(n_classes):
        m = y_true == c
        if m.any():
            scores.append(float((y_pred[m] == c).mean()))
    return float(np.mean(scores)) if scores else float("nan")


def class_weights(y_fit, n_classes):
    counts = np.bincount(y_fit, minlength=n_classes).astype(np.float64)
    weights = counts.sum() / (n_classes * np.maximum(counts, 1.0))
    return counts, weights


def main():
    cfg = json.loads(sys.argv[1])
    jobs = sys.argv[2].split(",")
    labels = cfg["labels"]
    n_classes = len(labels)
    dev = torch.device("cuda:0")

    random.seed(cfg["seed"])
    np.random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"])
    torch.cuda.manual_seed_all(cfg["seed"])

    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024 ** 3
    print(f"GPU {torch.cuda.get_device_name(0)} {mem_gb:.1f} GB  jobs={jobs}", flush=True)

    train = pd.read_csv(cfg["train_csv"])
    missing = [c for c in ("id", "prompt_id", "text", "model") if c not in train.columns]
    if missing:
        raise SystemExit(f"{cfg['train_csv']} is missing columns {missing}")
    train["text"] = train["text"].astype(str)
    train["model"] = train["model"].astype(str)
    unknown = sorted(set(train["model"]) - set(labels))
    if unknown:
        raise SystemExit(f"labels not in the 18-way list: {unknown}")
    train = train[train["model"].isin(labels)].reset_index(drop=True)
    y = train["model"].map({name: i for i, name in enumerate(labels)}).to_numpy()

    prompts = np.array(sorted(train["prompt_id"].unique()))
    rng0 = np.random.RandomState(cfg["fold_seed"])
    rng0.shuffle(prompts)
    fold = train["prompt_id"].map({p: i % 5 for i, p in enumerate(prompts)}).to_numpy()

    tok = AutoTokenizer.from_pretrained(cfg["model_dir"], local_files_only=True)
    tok.padding_side = "right"
    tok.model_max_length = int(1e9)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tokens = tok(
        train["text"].tolist(),
        add_special_tokens=True,
        truncation=True,
        max_length=cfg["max_len"],
    )["input_ids"]
    ok = np.array([len(t) >= 2 for t in tokens])
    if not ok.all():
        print(f"dropping {int((~ok).sum())} passages shorter than 2 tokens", flush=True)
        tokens = [t for t, keep in zip(tokens, ok) if keep]
        train = train.loc[ok].reset_index(drop=True)
        y = y[ok]
        fold = fold[ok]
    print(
        f"rows {len(train)}  prompts {train.prompt_id.nunique()}  "
        f"tokens median {int(np.median([len(t) for t in tokens]))} max {max(len(t) for t in tokens)}",
        flush=True,
    )

    def take(idxs):
        batch = tok.pad({"input_ids": [tokens[i] for i in idxs]}, return_tensors="pt")
        ids = batch["input_ids"].to(dev)
        attn = batch["attention_mask"].to(dev)
        yy = torch.as_tensor(y[list(idxs)], device=dev, dtype=torch.long)
        return ids, attn, yy

    def score_rows(model, idxs):
        """Both ATLAS readouts. logprob is mean token log p; prod is argmax-prod averaged over T."""
        model.eval()
        logprob = np.zeros((len(idxs), n_classes), np.float32)
        prod = np.zeros((len(idxs), n_classes), np.float32)
        order = np.argsort([len(tokens[i]) for i in idxs], kind="stable")
        bs = int(cfg["score_bs"])
        with torch.inference_mode():
            for start in range(0, len(order), bs):
                pos = order[start:start + bs]
                rows = [idxs[i] for i in pos]
                ids, attn, _ = take(rows)
                chunks = []
                for k in range(n_classes):
                    lab = torch.full((len(rows),), k, device=dev, dtype=torch.long)
                    with torch.autocast("cuda", dtype=torch.float16):
                        lp = model._theta_pass_single(ids, attn, lab)
                    chunks.append(lp.float())
                theta = torch.stack(chunks, dim=1)
                mask = attn[:, 1:].float()
                logprob[pos] = doc_score(theta, mask).float().cpu().numpy()
                prod[pos] = torch.stack([prod_readout(theta, mask, t) for t in PROD_TEMPS]).mean(0).cpu().numpy()
                del theta, chunks, ids, attn
        return logprob, prod

    for job in jobs:
        t0 = time.time()
        outdir = os.path.join(cfg["out_root"], f"job_{job}")
        os.makedirs(outdir, exist_ok=True)
        torch.manual_seed(cfg["seed"])
        torch.cuda.manual_seed_all(cfg["seed"])
        job_salt = 100 if job == "stage1" else (200 if job == "final" else int(job))
        rng = np.random.RandomState(cfg["seed"] + job_salt)

        if job in ("stage1", "final"):
            fit = np.arange(len(train))
            valid = np.array([], dtype=int)
        else:
            fit = np.where(fold != int(job))[0]
            valid = np.where(fold == int(job))[0]
        print(f"job {job} fit {len(fit)} valid {len(valid)}", flush=True)

        model = ATLASModel(
            cfg["model_dir"], labels, cfg["r"], cfg["alpha"], cfg["ldrop"], cfg["prefix_len"], dev,
        )
        if cfg.get("init_from"):
            model.load_weights(cfg["init_from"])
            print(f"job {job} loaded {cfg['init_from']}", flush=True)
        n_train = sum(p.numel() for p in model.trainable_parameters())
        print(f"job {job} trainable params {n_train:,}", flush=True)

        counts, weights = class_weights(y[fit], n_classes)
        class_w = None
        if cfg.get("class_weight"):
            class_w = torch.tensor(weights, dtype=torch.float32, device=dev)
            print(
                "class weights "
                + " ".join(f"{labels[i]}:{weights[i]:.2f}(n={int(counts[i])})" for i in range(n_classes)),
                flush=True,
            )

        params = model.trainable_parameters()
        opt = torch.optim.AdamW(params, lr=cfg["lr"], weight_decay=cfg["wd"], betas=(0.9, 0.999))
        # Steps are counted after the first epoch's batching; rebuild the schedule once the
        # per-epoch length is known. Every epoch here has the same number of microbatches.
        n_micro = int(math.ceil(len(fit) / float(cfg["batch_size"])))
        steps_per_epoch = max(1, int(math.ceil(n_micro / float(cfg["accum"]))))
        total_steps = steps_per_epoch * cfg["epochs"]
        warm = max(1, int(round(total_steps * cfg["warmup"])))

        def lr_factor(step):
            if step < warm:
                return float(step + 1) / float(warm)
            progress = min(1.0, float(step - warm) / float(max(1, total_steps - warm)))
            return 0.5 * (1.0 + math.cos(math.pi * progress))

        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_factor)
        scaler = torch.amp.GradScaler(init_scale=256.0)
        print(
            f"job {job} steps/epoch {steps_per_epoch} "
            f"total {total_steps} warmup {warm} lr {cfg['lr']} lam_sft {cfg['lam_sft']}",
            flush=True,
        )

        model.train()
        step = 0
        skipped = 0
        nan_streak = 0
        losses = []
        for ep in range(cfg["epochs"]):
            order = rng.permutation(fit)
            batches = [order[i:i + cfg["batch_size"]] for i in range(0, len(order), cfg["batch_size"])]
            opt.zero_grad(set_to_none=True)
            for i, batch_idx in enumerate(batches):
                ids, attn, yy = take(batch_idx)
                with torch.autocast("cuda", dtype=torch.float16):
                    out = model.compute_losses(ids, attn, yy, cfg["lam_sft"], class_w)
                loss = out["loss"]
                if not torch.isfinite(loss):
                    nan_streak += 1
                    opt.zero_grad(set_to_none=True)
                    if nan_streak >= 10:
                        raise RuntimeError(f"job {job} loss was non-finite for 10 steps (epoch {ep})")
                    continue
                nan_streak = 0
                scaler.scale(loss / cfg["accum"]).backward()
                losses.append(float(out["L_sft"]))
                if (i + 1) % cfg["accum"] == 0 or (i + 1) == len(batches):
                    scaler.unscale_(opt)
                    torch.nn.utils.clip_grad_norm_(params, cfg["grad_clip"])
                    before = scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()
                    if scaler.get_scale() < before:
                        skipped += 1
                    opt.zero_grad(set_to_none=True)
                    sched.step()
                    step += 1
                    if step % cfg["log_every"] == 0:
                        recent = float(np.mean(losses[-200:]))
                        print(
                            f"job {job} ep {ep} step {step}/{total_steps} "
                            f"nll {recent:.4f} loss {recent * cfg['lam_sft']:.4f} "
                            f"lr {sched.get_last_lr()[0]:.3e} skipped {skipped} ({time.time() - t0:.0f}s)",
                            flush=True,
                        )
                del ids, attn, yy, out, loss

        metrics = {
            "job": job,
            "steps": step,
            "skipped": skipped,
            "secs": time.time() - t0,
            "rows_fit": int(len(fit)),
            "rows_valid": int(len(valid)),
            "valid_bal_acc": None,
            "valid_bal_acc_prod": None,
            "valid_acc": None,
        }
        if len(valid):
            lp, pr = score_rows(model, valid)
            pred = lp.argmax(1)
            pred_prod = pr.argmax(1)
            yv = y[valid]
            metrics["valid_bal_acc"] = bal_acc(yv, pred, n_classes)
            metrics["valid_bal_acc_prod"] = bal_acc(yv, pred_prod, n_classes)
            metrics["valid_acc"] = float((pred == yv).mean())
            np.savez_compressed(
                os.path.join(outdir, "oof_scores.npz"),
                ids=train["id"].to_numpy()[valid],
                prompt_id=train["prompt_id"].to_numpy()[valid],
                model=train["model"].to_numpy()[valid],
                y=yv,
                logprob=lp,
                prod=pr,
                classes=np.array(labels),
            )
            print(
                f"job {job} valid balanced acc argmax {metrics['valid_bal_acc']:.4f} "
                f"prod {metrics['valid_bal_acc_prod']:.4f} plain {metrics['valid_acc']:.4f}",
                flush=True,
            )

        model.save_weights(outdir)
        with open(os.path.join(outdir, "hyperparams.json"), "w") as f:
            json.dump(cfg, f, indent=2)
        metrics["secs"] = time.time() - t0
        with open(os.path.join(outdir, "COMPLETE.json"), "w") as f:
            json.dump(metrics, f, indent=2)
        print(
            f"job {job} done steps {step} skipped {skipped} ({metrics['secs']:.0f}s) -> {outdir}",
            flush=True,
        )
        del model, opt, sched, scaler, params, class_w
        import gc
        gc.collect()
        torch.cuda.empty_cache()


if __name__ == "__main__":
    main()


## Locate Qwen2.5-7B and launch

Finds host `train.csv` and a local Qwen2.5-7B checkpoint under `/kaggle/input`, then starts one process per GPU. Each process trains its jobs from the base model. Finished jobs are left in place; delete `lora_out/job_<id>` before relaunching if you want that job trained again.


In [ ]:
def find_train_csv() -> pathlib.Path:
    for p in [
        pathlib.Path("/kaggle/input/competitions/name-that-llm/train.csv"),
        pathlib.Path("/kaggle/input/name-that-llm/train.csv"),
        pathlib.Path("data/train.csv"),
        pathlib.Path("../data/train.csv"),
    ]:
        if p.exists():
            return p
    raise FileNotFoundError("train.csv not found. Attach the name-that-llm competition.")


def find_qwen25() -> str:
    hits = []
    for cfg_path in sorted(glob.glob("/kaggle/input/**/config.json", recursive=True)):
        try:
            j = json.load(open(cfg_path))
        except Exception:
            continue
        if j.get("model_type") != "qwen2":
            continue
        if j.get("hidden_size") != 3584 or j.get("num_hidden_layers") != 28:
            continue
        folder = os.path.dirname(cfg_path)
        low = folder.lower()
        if "qwen2.5" not in low and "qwen2-5" not in low and "qwen2_5" not in low:
            continue
        if any(bad in low for bad in ("awq", "gptq", "int4", "int8")):
            continue
        hits.append(folder)
    if not hits:
        raise FileNotFoundError(
            "Qwen2.5-7B not found under /kaggle/input. Add the Kaggle model qwen-lm/qwen2.5, variation 7b."
        )

    def rank(path):
        low = path.lower()
        return (1 if "instruct" in low else 0, 1 if "chat" in low else 0, len(path))

    hits.sort(key=rank)
    return hits[0]


T0 = time.time()
host_csv = str(find_train_csv())
model_dir = find_qwen25()
print("train", host_csv, flush=True)
print("base", model_dir, flush=True)
assert pathlib.Path("worker.py").exists() and "class ATLASModel" in pathlib.Path("worker.py").read_text(encoding="utf-8"), (
    "Run both worker writefile cells first"
)

cfg = dict(
    stage=1,
    train_csv=host_csv,
    model_dir=model_dir,
    labels=LABELS,
    fold_seed=FOLD_SEED,
    seed=SEED,
    max_len=MAX_LEN,
    prefix_len=PREFIX_LEN,
    lam_sft=LAM_SFT,
    warmup=WARMUP,
    wd=WD,
    grad_clip=GRAD_CLIP,
    r=R,
    alpha=ALPHA,
    ldrop=LDROP,
    score_bs=SCORE_BS,
    log_every=LOG_EVERY,
    out_root=OUT_ROOT,
    epochs=EPOCHS,
    lr=LR,
    batch_size=BATCH,
    accum=ACCUM,
    class_weight=CLASS_WEIGHT,
    init_from="",
)

pending = {}
for gpu, job_list in JOBS.items():
    left = [j for j in job_list if not (pathlib.Path(OUT_ROOT) / f"job_{j}" / "COMPLETE.json").exists()]
    done_jobs = [j for j in job_list if j not in left]
    if done_jobs:
        print(f"GPU {gpu} already complete: {done_jobs}", flush=True)
    if left:
        pending[gpu] = left
if not pending:
    print("every job already has COMPLETE.json; delete lora_out/job_<id> to retrain", flush=True)

procs = []
for gpu, job_list in pending.items():
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")
    log_path = f"worker_g{gpu}.log"
    print(f"launch GPU {gpu} jobs {job_list}", flush=True)
    procs.append((
        gpu,
        log_path,
        subprocess.Popen(
            [sys.executable, "worker.py", json.dumps(cfg), ",".join(job_list)],
            env=env,
            stdout=open(log_path, "w"),
            stderr=subprocess.STDOUT,
        ),
    ))
failed = False
for gpu, log_path, proc in procs:
    proc.wait()
    print(f"---- worker {gpu} (rc {proc.returncode})")
    text = open(log_path, encoding="utf-8", errors="replace").read()
    print(text[-5000:])
    if proc.returncode != 0:
        failed = True
if failed:
    raise SystemExit("a worker failed")

done = sorted(glob.glob(f"{OUT_ROOT}/job_*/COMPLETE.json"))
summary = pd.DataFrame([json.load(open(p)) for p in done])
cols = ["job", "valid_bal_acc", "valid_bal_acc_prod", "valid_acc", "steps", "skipped", "secs", "rows_fit", "rows_valid"]
if not summary.empty:
    summary = summary[[c for c in cols if c in summary.columns]]
    print(summary.to_string(index=False))
print(f"total wall time: {time.time() - T0:.0f}s")
pd.DataFrame({"id": [], "model": []}).to_parquet("submission.parquet", index=False)
